# 数据加载与导出 (I/O)

## 读取数据

In [ ]:
import pandas as pd

# 读取 CSV (最常用)
df = pd.read_csv('data.csv')

# 读取 JSON (LLM 数据集常以此格式存储)
# orient='records' 表示每一行是一个独立的 JSON 对象
df_json = pd.read_json('data.json',orient='records')

# 读取 Excel
df_excel = pd.read_excel('data.xlsx',sheet_name='Sheet1')

# 读取 Parquet (大规模数据存储格式)
df_parquet = pd.read_parquet('large_dataset.parquet')

### read_csv核心参数
| 参数 | 说明 | 常用值/示例 | LLM 应用场景 |
| :--- | :--- | :--- | :--- |
| `filepath_or_buffer` | 文件路径或对象 | `'data.csv'`, `StringIO` | 指定数据源 |
| `sep` / `delimiter` | 指定分隔符 | `','` (默认), `'\t'` (TSV), `'|'` | 处理非标准 CSV (如管道符分隔) |
| `encoding` | 文件编码格式 | `'utf-8'` (默认), `'gbk'`, `'latin-1'` | 解决中文乱码 (Windows 导出文件常用 gbk) |
| `header` | 指定哪一行作为列名 | `0` (默认第一行), `None` (无列名) | 处理没有表头的原始数据 |
| `names` | 自定义列名列表 | `['input', 'output', 'label']` | 当 `header=None` 时，必须指定列名 |
| `index_col` | 指定某列为索引 | `0` (第一列), `'id'` | 将 ID 列设为索引，方便后续查找 |
| `usecols` | 只读取指定的列 | `['text', 'label']` 或 `[0, 1]` | 内存优化：只加载模型需要的列，忽略无关列 |
| `dtype` | 指定列的数据类型 | `{'id': 'str', 'score': 'float32'}` | 防止 ID 变成数字，或节省内存 (float32 vs float64) |
| `na_values` | 指定哪些值算作缺失值 | `['NA', 'null', '?', '']` | 统一数据中的缺失标记 |
| `chunksize` | 分块读取，返回迭代器 | `1000` (整数) | 超大文件处理：处理 GB 级训练数据 |
| `nrows` | 只读取前 N 行 | `100` | 调试：快速加载少量数据测试代码逻辑 |
| `parse_dates` | 解析日期列 | `['timestamp']` | 处理带有时间戳的对话数据 |

示例：
```python
df = pd.read_csv(
    'data.csv',
    encoding='utf-8',      # 或者是 'gbk'
    usecols=['text', 'label'], # 只读需要的列
    dtype={'text': str},   # 保证文本列不被破坏
    na_values=['NA', '']   # 统一缺失值
)
```

### read_json核心参数
| 参数 | 说明 | 常用值/示例 | LLM 应用场景 |
| :--- | :--- | :--- | :--- |
| `orient` | 指示 JSON 的结构格式 | `'records'`, `'split'`, `'index'`, `'columns'`, `'values'` | 处理不同 API 返回的数据结构 |
| `lines` | 是否逐行读取 (JSON Lines 格式) | `True` / `False` | 非常重要：读取大规模预训练语料 (如 Common Crawl) |
| `dtype` | 强制指定列的数据类型 | `{'col_name': 'str'}` | 防止 ID 列被自动转为数字，节省内存 |
| `convert_dates` | 是否尝试解析日期 | `True` / `False` | 处理带有时间戳的对话数据 |
| `chunksize` | 分块读取，返回迭代器 | `1000` (整数) | 内存优化：处理无法一次性装入内存的超大 JSON 文件 |

示例：
```python
df_json = pd.read_json(
    'data.json',
    orient='records',  # 每行一个 JSON 对象
    lines=True,        # 逐行读取
    dtype={'id': str}  # 确保 ID 列不被破坏
)
```

### read_excel核心参数
| 参数 | 说明 | 常用值/示例 | LLM 应用场景 |
| :--- | :--- | :--- | :--- |
| `io` | 文件路径、URL 或文件对象 | `'data.xlsx'` | 指定文件源 |
| `sheet_name` | 指定工作表 | `0`, `'Sheet1'`, `['S1', 'S2']`, `None` (读所有) | 整合多个标注员提交的 Sheet |
| `header` | 指定哪一行作为列名 | `0` (默认), `None` (无列名), `[0, 1]` (多级表头) | 处理包含复杂标题行的报表 |
| `index_col` | 指定某列为索引 | `0`, `'id'` | 将样本 ID 设为索引 |
| `usecols` | 只读取指定的列 | `'A:C'`, `[0, 2]`, `['Prompt', 'Response']` | 性能优化：忽略 Excel 中的格式列或备注列 |
| `nrows` | 只读取前 N 行 | `100` | 调试：快速验证数据清洗脚本 |
| `skiprows` | 跳过文件开头的行数 | `2`, `lambda x: x in [0, 2]` | 跳过 Excel 顶部的说明文字或元数据 |
| `dtype` | 指定数据类型 | `{'id': str}` | 防止 ID 列（如 '001'）丢失前导零 |
| `engine` | 指定解析引擎 | `'openpyxl'`, `'xlrd'` | 关键：处理不同版本的 Excel 文件 |

示例：
```python
df = pd.read_excel(
    'dataset.xlsx',
    sheet_name='Sheet1',
    usecols=['Input_Text', 'Target_Label'], # 只读关键列
    dtype={'ID': str},                       # 保持 ID 格式
    engine='openpyxl'                         # 显式指定引擎
)
```

### read_parquet核心参数
| 参数 | 说明 | 常用值/示例 | LLM 应用场景 |
| :--- | :--- | :--- | :--- |
| `path` | 文件路径、目录或 URL | `'data.parquet'`, `'s3://...'` | 支持本地文件、HDFS、S3 等 |
| `engine` | 读取引擎 | `'pyarrow'` (推荐), `'fastparquet'` | 推荐显式指定 `engine='pyarrow'` 以确保性能 |
| `columns` | 只读取指定的列 | `['input_ids', 'attention_mask']` | 核心加速：训练只需文本列，忽略标签列 |
| `filters` | 行级过滤 | `[('date', '=', '2023-10-01')]` | 谓词下推：只加载特定日期的数据，不读全表 |
| `dtype_backend` | 数据类型后端 | `'numpy_nullable'`, `'pyarrow'` | 解决 Pandas 默认类型（如 int64）无法处理 NaN 的问题 |
| `storage_options` | 云存储配置 | `{'key': '...', 'secret': '...'}` | 读取 S3、GCS 等云存储文件时的鉴权信息 |

示例：
```python
df = pd.read_parquet(
    'dataset.parquet',
    engine='pyarrow',          # 显式指定引擎
    columns=['text', 'label'], # 只读关键列（极大提升速度）
    dtype_backend='pyarrow'    # 使用 Arrow 类型系统（更稳健）
)
```

## 导出数据

In [ ]:
# 导出为 CSV，index=False 避免保存行索引
df.to_csv('cleaned_data.csv',index=False)

# 导出为 JSON
df.to_json('output.json',orient='records',force_ascii=False) # force_ascii=False 保持中文字符不被转义

### df.to_csv核心参数
| 参数 | 说明 | 常用值/示例 | LLM 应用场景 |
| :--- | :--- | :--- | :--- |
| `path_or_buf` | 文件路径或文件对象 | `'data.csv'`, `None` | 指定保存位置；若为 `None` 则返回字符串 |
| `index` | 是否写入行索引 | `False` (推荐), `True` | 必用：防止保存多余的索引列，保持数据整洁 |
| `encoding` | 文件编码格式 | `'utf-8'` (默认), `'utf-8-sig'`, `'gbk'` | 解决乱码：`utf-8-sig` 可防止 Excel 打开中文乱码 |
| `sep` | 分隔符 | `','` (默认), `'\t'` (TSV) | 导出 TSV 格式，常用于 NLP 数据交换 |
| `columns` | 指定要保存的列 | `['text', 'label']` | 只导出模型需要的特定列，忽略中间计算列 |
| `mode` | 文件写入模式 | `'w'` (覆盖), `'a'` (追加) | 日志记录：将多次推理结果追加到同一个文件 |
| `compression` | 文件压缩格式 | `'gzip'`, `'zip'` | 节省空间：直接保存为 `.csv.gz`，减小文件体积 |
| `na_rep` | 缺失值的表示 | `'NULL'`, `''` (空字符串) | 统一处理 NaN 值，防止模型读取时报错 |
| `chunksize` | 分块写入行数 | `10000` | 内存优化：处理超大 DataFrame 时防止内存溢出 |

示例：
```python
df.to_csv(
    'output.csv',
    index=False,           # 不保存索引
    encoding='utf-8-sig',  # 兼容 Excel 的中文编码
    sep=','                # 标准逗号分隔
)
```

### df.to_json核心参数
| 参数 | 说明 | 常用值/示例 | LLM 应用场景 |
| :--- | :--- | :--- | :--- |
| `path_or_buf` | 文件路径或文件对象 | `'data.json'`, `None` | 指定保存位置；若为 `None` 则返回字符串 |
| `orient` | 最重要。定义 JSON 的结构格式 | `'records'`, `'split'`, `'index'`, `'columns'`, `'table'` | 决定数据是“行优先”还是“列优先”，适配不同 API |
| `force_ascii` | 是否强制转义非 ASCII 字符 | `True` (默认), `False` | 解决中文乱码：设为 `False` 以保留原始中文字符 |
| `indent` | 缩进空格数 | `4`, `None` (紧凑) | 调试阅读：设为 4 可生成格式化的美观 JSON |
| `index` | 是否包含索引 | `True`, `False` | 在 `orient='split'` 或 `'table'` 时控制是否写入索引 |
| `lines` | 是否按行输出 | `True`, `False` | 大数据处理：生成 JSON Lines (`.jsonl`) 格式，每行一个对象 |
| `date_format` | 日期时间格式 | `'iso'`, `'epoch'` | 处理时间序列数据时，统一日期格式 |
| `compression` | 文件压缩格式 | `'gzip'`, `'zip'` | 节省空间：直接保存为 `.json.gz` |

示例：
```python
df.to_json(
    'output.json',
    orient='records',      # 列表字典格式
    lines=True,            # 如果是为了生成 .jsonl 训练数据
    force_ascii=False,     # 保留中文
    indent=4               # 如果是给人看的，加上缩进
)
```

# 数据清洗

## 处理缺失值 (NaN)

In [ ]:
# 检查缺失值数量
print(df.isnull().sum())

# 删除包含缺失值的行 (简单粗暴)
df.dropna(inplace=True)

# 填充缺失值 (常用策略)
# 文本列填"Unknown"，数值列填0或均值
df['text_column']=df['text_column'].fillna('Unknown')
df['score']=df['score'].fillna(df['score'].mean())

## 处理重复值

In [ ]:
# 检查重复行数量
print(df.duplicated().sum())

# 删除完全重复的行
df.drop_duplicates(inplace=True)

# 根据特定列去重 (例如：保留ID相同的最新一条数据)
df.drop_duplicates(subset=['id'],keep='last',inplace=True)

## 数据类型转换

In [ ]:
# 转换列类型
df['age'] = df['age'].astype(int) # 转为整数
df['price'] = df['price'].astype(float) # 转为浮点数

# 将对象类型转换为类别类型 (Categorical)
# 在标签分类任务中，这能极大节省内存
df['label'] = df['label'].astype('category') # 转为类别类型